In [ ]:
!pip install -q nnunetv2 simpleitk scikit-image tqdm pyyaml

!python kaggle_helper.py download-competition \
    --competition vesuvius-challenge-surface-detection \
    --out data/

!python kaggle_helper.py download-dataset \
    --dataset p4rallax/vesuvius-coarse-nnunet-baseline \
    --out nnunet_results/

!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw \
 NNUNet_preprocessed=./nnunet/preprocessed \
 NNUNet_results=./nnunet/nnUNet_results \
 python nnUNet_utils/build_nnunet_dataset.py

!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw \
 NNUNet_preprocessed=./nnunet/preprocessed \
 NNUNet_results=./nnunet/nnUNet_results \
 python nnUNet_utils/train_nnunet.py

# 6) Optional: explicit per‑fold training to save validation softmax (.npz) OOF
!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw NNUNet_preprocessed=./nnunet/preprocessed NNUNet_results=./nnunet/nnUNet_results python -m nnunetv2.run.run_training 900 3d_fullres 0 -num_gpus 1 -p nnUNetResEncUNetMPlans --val --npz
!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw NNUNet_preprocessed=./nnunet/preprocessed NNUNet_results=./nnunet/nnUNet_results python -m nnunetv2.run.run_training 900 3d_fullres 1 -num_gpus 1 -p nnUNetResEncUNetMPlans --val --npz
!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw NNUNet_preprocessed=./nnunet/preprocessed NNUNet_results=./nnunet/nnUNet_results python -m nnunetv2.run.run_training 900 3d_fullres 2 -num_gpus 1 -p nnUNetResEncUNetMPlans --val --npz
!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw NNUNet_preprocessed=./nnunet/preprocessed NNUNet_results=./nnunet/nnUNet_results python -m nnunetv2.run.run_training 900 3d_fullres 3 -num_gpus 1 -p nnUNetResEncUNetMPlans --val --npz
!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw NNUNet_preprocessed=./nnunet/preprocessed NNUNet_results=./nnunet/nnUNet_results python -m nnunetv2.run.run_training 900 3d_fullres 4 -num_gpus 1 -p nnUNetResEncUNetMPlans --val --npz

# 7) Merge OOF softmax files (avoid manual moving)
!python - << 'PY'
from pathlib import Path
import shutil
oof = Path("./nnunet/nnUNet_results/Dataset900_VesuviusScroll/nnUNetTrainer__nnUNetResEncUNetMPlans__3d_fullres/oof_softmax")
oof.mkdir(parents=True, exist_ok=True)
for i in range(5):
    fold = oof / f"fold{i}"
    if fold.exists():
        for f in fold.glob("*.npz"):
            f.rename(oof / f.name)
        shutil.rmtree(fold, ignore_errors=True)
print("OOF npz files:", len(list(oof.glob("*.npz"))))
PY

# 8) Build nnUNet soft OOF for DeformNet3D
!NNUNet_raw=./nnunet/nnUNet_raw_data_base/nnUNet_raw \
 NNUNet_preprocessed=./nnunet/preprocessed \
 NNUNet_results=./nnunet/nnUNet_results \
 python nnUNet_utils/generate_nnunet_soft_oof.py

# 9) Write configs/config_deform.yaml (the 4 fields from README)
!python - << 'PY'
import yaml, os
from pathlib import Path
Path("./configs").mkdir(parents=True, exist_ok=True)
cfg = {
    "data_path": "./data",
    "nnunet_path": "./nnunet/nnUNet_results",
    "petrained_ckpt_path": "",  # set if you have a pretrained DeformNet checkpoint
    "data_split_path": "./nnunet/nnUNet_results/Dataset900_VesuviusScroll/splits_final.json"
}
with open("./configs/config_deform.yaml", "w") as f:
    yaml.safe_dump(cfg, f)
print(open("./configs/config_deform.yaml").read())
PY

!python train_deformnet.py

!python - << 'PY'
import json, time, shutil
from pathlib import Path
artifacts_dir = Path("./artifacts"); artifacts_dir.mkdir(exist_ok=True)
stamp = time.strftime("%Y%m%d-%H%M%S")
src = Path("./nnunet/nnUNet_results")
zip_path = artifacts_dir / f"vesuvius-models-v2_{stamp}.zip"
shutil.make_archive(str(zip_path).replace(".zip",""), "zip", src)
manifest = {
    "name": "vesuvius-models-v2",
    "created": stamp,
    "competition": "vesuvius-challenge-surface-detection",
    "dataset_id": 900,
    "config": "3d_fullres",
    "plans": "nnUNetResEncUNetMPlans",
    "results_dir": str(src),
    "zip": str(zip_path)
}
Path(artifacts_dir / "MANIFEST.json").write_text(json.dumps(manifest, indent=2))
print("Saved:", zip_path)
print("Manifest:", (artifacts_dir / "MANIFEST.json").read_text())
PY

print("✅ Retrain pipeline completed locally.")